# Lab 4 - Data Quality, Reliability, and Maintenance

Verifies quality rules, rerun safety, Delta history, clustering, OPTIMIZE, and VACUUM.


In [0]:
%run ./lab4_00_config


In [0]:
from pyspark.sql import functions as F

silver_df = spark.table(silver_curated_table)

quality_results = silver_df.select(
    F.sum(F.when(F.col("show_id").isNull() | (F.length(F.trim(F.col("show_id"))) == 0), 1).otherwise(0)).alias("invalid_show_id"),
    F.sum(F.when(F.col("title").isNull() | (F.length(F.trim(F.col("title"))) == 0), 1).otherwise(0)).alias("invalid_title"),
    F.sum(F.when(~F.coalesce(F.col("content_type").isin("Movie", "TV Show"), F.lit(False)), 1).otherwise(0)).alias("invalid_content_type"),
    F.sum(F.when(~F.coalesce(F.col("release_year").between(1900, F.year(F.current_date()) + 1), F.lit(False)), 1).otherwise(0)).alias("invalid_release_year"),
    F.sum(F.when(F.col("duration_value").isNotNull() & (F.col("duration_value") <= 0), 1).otherwise(0)).alias("invalid_duration")
)

display(quality_results)


In [0]:
duplicate_show_ids = (
    spark.table(silver_curated_table)
    .groupBy("show_id")
    .count()
    .filter(F.col("count") > 1)
)

history_current_duplicates = (
    spark.table(silver_history_table)
    .filter(F.col("is_current") == True)
    .groupBy("show_id")
    .count()
    .filter(F.col("count") > 1)
)

print("Curated duplicate show_id rows:", duplicate_show_ids.count())
print("History current duplicate show_id rows:", history_current_duplicates.count())


In [0]:
display(spark.sql(f"DESCRIBE DETAIL {silver_curated_table}").select("numFiles", "sizeInBytes", "format"))
display(spark.sql(f"DESCRIBE HISTORY {silver_curated_table}"))


In [0]:
try:
    spark.sql(f"ALTER TABLE {silver_curated_table} CLUSTER BY (show_id, content_type)")
    spark.sql(f"OPTIMIZE {silver_curated_table}")
    spark.sql(f"OPTIMIZE {silver_history_table}")
except Exception as exc:
    print("Maintenance optimization command was not applied in this workspace/runtime:")
    print(str(exc)[:1000])


In [0]:
# Keep the default 7-day safety window.
try:
    spark.sql(f"VACUUM {silver_curated_table} RETAIN 168 HOURS")
    spark.sql(f"VACUUM {silver_history_table} RETAIN 168 HOURS")
except Exception as exc:
    print("VACUUM was not applied in this workspace/runtime:")
    print(str(exc)[:1000])


## Scheduling

Databricks Job was created here: 